# Fake News Detection - LSTM Model (Jupyter Notebook)

This notebook trains a Bidirectional LSTM model to detect fake news and provides interactive predictions.

## Step 1: Install Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'tensorflow>=2.12.0',
    'numpy>=1.23.0',
    'pandas>=1.5.0',
    'scikit-learn>=1.2.0',
    'nltk>=3.8.0',
    'matplotlib>=3.6.0',
    'seaborn>=0.12.0'
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("✓ All dependencies installed!")

## Step 2: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import re
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Deep Learning
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print("✓ All libraries imported successfully!")

## Step 3: Configuration

In [ ]:
# Configuration
MAX_VOCAB_SIZE = 10000
MAX_SEQUENCE_LEN = 300
EMBEDDING_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE = 0.3
BATCH_SIZE = 64
EPOCHS = 10
TEST_SIZE = 0.2
RANDOM_STATE = 42

OUTPUT_DIR = "model_artifacts"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

## Step 4: Load & Explore Data

In [ ]:
# Load datasets
print("Loading datasets...")
true_df = pd.read_csv('True.csv')
fake_df = pd.read_csv('Fake.csv')

print(f"✓ True news: {len(true_df)} articles")
print(f"✓ Fake news: {len(fake_df)} articles")

# Combine and label
true_df['label'] = 1  # Real
fake_df['label'] = 0  # Fake

# Combine datasets
df = pd.concat([true_df, fake_df], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"\n✓ Total articles: {len(df)}")
print(f"✓ Label distribution:\n{df['label'].value_counts()}")
print(f"\nSample data:\n{df.head()}")

## Step 5: Data Preprocessing

In [ ]:
# Initialize stemmer and stopwords
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """Clean and preprocess text"""
    if pd.isna(text):
        return ""
    # Convert to lowercase
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    words = text.split()
    # Remove stopwords and stem
    words = [stemmer.stem(w) for w in words if w not in stop_words]
    return ' '.join(words)

print("Preprocessing text...")
df['text'] = df['title'] + ' ' + df['text']  # Combine title and text
df['text'] = df['text'].apply(preprocess_text)

print("✓ Preprocessing complete!")
print(f"Sample preprocessed text:\n{df['text'].iloc[0][:200]}...")

## Step 6: Tokenization & Padding

In [ ]:
print("Tokenizing text...")
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df['text'])

# Pad sequences
X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LEN, padding='post')
y = df['label'].values

print(f"✓ Tokenization complete!")
print(f"  - Vocabulary size: {len(tokenizer.word_index)}")
print(f"  - Sequence shape: {X.shape}")

# Save tokenizer
with open(os.path.join(OUTPUT_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"✓ Tokenizer saved!")

## Step 7: Train-Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"✓ Train-test split complete!")
print(f"  - Training set: {X_train.shape}")
print(f"  - Test set: {X_test.shape}")
print(f"  - Train labels: {np.bincount(y_train)}")
print(f"  - Test labels: {np.bincount(y_test)}")

## Step 8: Build LSTM Model

In [ ]:
print("Building LSTM model...")
model = Sequential([
    Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LEN),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True)),
    Dropout(DROPOUT_RATE),
    LSTM(32),
    Dropout(DROPOUT_RATE),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print("✓ Model built!")
print(model.summary())

## Step 9: Train the Model

In [ ]:
print("Training model (this may take 10-20 minutes)...\n")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'lstm_fake_news_model.keras'),
        monitor='val_accuracy',
        save_best_only=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Model training complete!")

## Step 10: Evaluate Model

In [ ]:
print("Evaluating model...\n")

# Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Metrics
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

# Save metrics
metrics = {
    'accuracy': float(accuracy),
    'confusion_matrix': cm.tolist()
}
with open(os.path.join(OUTPUT_DIR, 'metrics.json'), 'w') as f:
    import json
    json.dump(metrics, f)

## Step 11: Visualize Results

In [ ]:
# Training history plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=100)
plt.show()

print("✓ Training history plot saved!")

In [ ]:
# Confusion Matrix plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake', 'Real'],
            yticklabels=['Fake', 'Real'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=100)
plt.show()

print("✓ Confusion matrix plot saved!")

## Step 12: Make Predictions on New Text

In [ ]:
def predict_news(text):
    """Predict if news is fake or real"""
    # Preprocess
    processed = preprocess_text(text)
    # Tokenize
    sequence = tokenizer.texts_to_sequences([processed])
    # Pad
    padded = pad_sequences(sequence, maxlen=MAX_SEQUENCE_LEN, padding='post')
    # Predict
    prediction = model.predict(padded, verbose=0)
    confidence = prediction[0][0]
    label = 'REAL' if confidence > 0.5 else 'FAKE'
    return label, confidence

# Test with sample news
sample_texts = [
    "Scientists discover new treatment for cancer",
    "This shocking secret will change your life forever!!!!"
]

print("Testing predictions on sample texts:\n")
for text in sample_texts:
    label, confidence = predict_news(text)
    print(f"Text: {text[:60]}...")
    print(f"Prediction: {label} (Confidence: {confidence:.4f})\n")

## Step 13: Interactive Prediction

In [ ]:
# Enter your own news text here
my_news = "Enter your news article text here"

label, confidence = predict_news(my_news)
print(f"\nPrediction: {label}")
print(f"Confidence: {confidence:.4f}")

if label == 'REAL':
    print("✓ This appears to be REAL news")
else:
    print("⚠ This appears to be FAKE news")

## Summary

✓ Model trained successfully!  
✓ Artifacts saved in `model_artifacts/`:
- `lstm_fake_news_model.keras` - Trained model
- `tokenizer.pkl` - Tokenizer
- `training_history.png` - Training curves
- `confusion_matrix.png` - Confusion matrix
- `metrics.json` - Performance metrics

To use the model in the Streamlit app, run: `streamlit run app.py`